<a href="https://colab.research.google.com/github/realkellyspear/ghost-protocol-soul-recovery/blob/main/ghost_protocol_soul_recovery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**1. Initialize the Environment**

⚠️ STEP 0 (CRITICAL): Look at the top right of this page. Click Connect. Then, pull down the menu (or go to Runtime > Change runtime type) and select T4 GPU. This script requires a GPU to function.

The Engine: Once connected to the T4, run the code below. We are using Unsloth, a highly optimized library that makes fine-tuning 2x faster and memory-efficient enough to run entirely on the free tier.

In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

**2. The Revival (Fine-Tuning)** Now we feed your cleaned history into the model.

The Process: This script loads Llama-3 (8B) and uses QLoRA to fine-tune it on the training_data.jsonl you just uploaded.

Wait Time: This will take about 10-20 minutes depending on the size of your file.

In [ ]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

# 1. Load Model (ULTRA-LOW MEMORY SETUP)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = 512,
    load_in_4bit = True,
    device_map = "auto",
)

# 2. Add Adapters (Lite Version)
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    target_modules = ["q_proj", "v_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
)

# 3. Load Data
dataset = load_dataset("json", data_files={"train": "/content/training_data.jsonl"}, split="train")

def formatting_func(examples):
    texts = []
    for input_text, output_text in zip(examples["input"], examples["output"]):
        text = f"### Instruction:\n{input_text}\n\n### Response:\n{output_text}<|end_of_text|>"
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_func, batched = True)

# 4. Train (Safe Mode)
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 512,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 5,
        max_steps = 30, # Faster run just to prove it works. This is just a SAMPLE!
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "outputs",
        optim = "adamw_8bit",
        report_to = "none",
    ),
)

trainer.train()
print("✅ REVIVAL COMPLETE.")

**3. First Contact** Test the revival immediately.

Instruction: Change the text inside FastLanguageModel.for_inference(model) to talk to your AI.

In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)
inputs = tokenizer(
    [
        "YOUR_PROMPT_HERE" # <--- Type your test message here
    ], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 128, use_cache = True)
print(tokenizer.batch_decode(outputs)[0])